In [3]:
import os
import amsterdamumcdb as adb
import pandas as pd
from google.cloud import bigquery

#login terminal commadns
#brew install --cask google-cloud-sdk   # skip if already installedd
#gcloud auth application-default login
#gcloud auth application-default set-quota-project liquid-folio-513114-d9

projectId = "liquid-folio-513114-d9"
datasetId = "amsterdamumcdb.version1_5_0"
location = "eu"
searchPattern = "extub|detub|reintub|intub"

os.environ["GOOGLE_CLOUD_PROJECT"] = projectId
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 200)

#default dataset
#fail above 20gb to be sure and not burn everythin
jobConfig = bigquery.QueryJobConfig(default_dataset=datasetId, maximum_bytes_billed=20*1024**3)
client = bigquery.Client(project=projectId, location=location, default_query_job_config=jobConfig)

In [4]:
person = client.query("SELECT * FROM person").to_dataframe()
print(f"Number of persons: {len(person)}")

Number of persons: 20109


Extubation concepts


In [5]:
#Local check for the extrbation concepts
dictionary = adb.get_dictionary()
inDescription = dictionary["source_code_description"].str.contains(searchPattern, case=False, na=False)
inConceptName = dictionary["concept_name"].str.contains(searchPattern, case=False, na=False)
dictionaryHits = dictionary[inDescription | inConceptName]
print(f"\n!!!!!!!!!Dictionary rows matching '{searchPattern}': {len(dictionaryHits)}")
print(dictionaryHits[["source_vocabulary_id", "source_code", "source_code_description", "concept_id", "concept_name", "source_frequency"]])




!!!!!!!!!Dictionary rows matching 'extub|detub|reintub|intub': 21
       source_vocabulary_id source_code                            source_code_description  concept_id                              concept_name  source_frequency
3135              AUMC List       14562                             Intubatie_Complicaties         NaN                                       NaN               125
3150              AUMC List       14577                                 Intubatie Techniek   4202832.0                                Intubation              2111
3483              AUMC List       12623                                        Tube diepte   3046004.0  Intubation tube depth Respiratory system             17394
3765        AUMC Meas Value     19637-9                                        Geïntubeerd  36311192.0                                 Intubated               813
3807        AUMC Meas Value    19640-15                                        Geïntubeerd  36311192.0                

In [ ]:
#These are some things we found from website and above. Used in this notebook to see what vars we want to keep/iuse
#Found ventilation mode in measurements, it shows what mode of ventilation is done
ventilationModeConceptId = 3004921
ventilationModeSources = ["Ventilatie Mode (Set)", "Ventilatie Mode (Set) (2)", "Type Beademing Evita 4", "Type Beademing Evita 4(2)", "Type beademing Evita 1", "Mode (Bipap Vision)"]

#gcs has text in it with intubation as answer if intubated (not very complete we found latr)
gcsVerbalConceptId = 3009094
gcsVerbalSource = "Beste verbale reactie"
intubatedValueConceptId = 36311192

#intubation table procedure_occurrence
intubationConceptId = 4202832
intubationSources = ["Intuberen", "Intubatie Techniek", "Intubatie"]

#extubation order table procedure_occurrence (but order so mauybe only the operative patients?) - seems to be due to dates that NICE didnt record a lot in 2013 cohort
extubationConceptId = 4150627
extubationSource = "Extuberen"

#self extubation so should skip tose patients if we find. (misses a lot but still used if present)
autoExtubationSource = "Autodetubatie - Sedatie uit uren"

how complete each item records admissions and date range

In [ ]:
#Overview of above, is just copy paste of one query into the next. Just to get a feeling how many admissions and records and dates
#After each union all is a distinct section containing the concepts abocve for fast and complete check this query
overviewSql = f"""
SELECT 'ventilation mode' AS item, measurement_source_value AS sourceValue, COUNT(*) AS records,
  COUNT(DISTINCT visit_occurrence_id) AS admissions, MIN(measurement_datetime) AS firstDate, MAX(measurement_datetime) AS lastDate
FROM measurement
WHERE provider_id IS NOT NULL AND measurement_concept_id = {ventilationModeConceptId}
GROUP BY measurement_source_value
UNION ALL
SELECT 'gcs verbal', measurement_source_value, COUNT(*), COUNT(DISTINCT visit_occurrence_id), MIN(measurement_datetime), MAX(measurement_datetime)
FROM measurement
WHERE provider_id IS NOT NULL AND measurement_concept_id = {gcsVerbalConceptId}
GROUP BY measurement_source_value
UNION ALL
SELECT 'gcs verbal intubated', measurement_source_value, COUNT(*), COUNT(DISTINCT visit_occurrence_id), MIN(measurement_datetime), MAX(measurement_datetime)
FROM measurement
WHERE provider_id IS NOT NULL AND measurement_concept_id = {gcsVerbalConceptId} AND value_as_concept_id = {intubatedValueConceptId}
GROUP BY measurement_source_value
UNION ALL
SELECT 'intubation', procedure_source_value, COUNT(*), COUNT(DISTINCT visit_occurrence_id), MIN(procedure_datetime), MAX(procedure_datetime)
FROM procedure_occurrence
WHERE provider_id IS NOT NULL AND procedure_concept_id = {intubationConceptId}
GROUP BY procedure_source_value
UNION ALL
SELECT 'extubation order', procedure_source_value, COUNT(*), COUNT(DISTINCT visit_occurrence_id), MIN(procedure_datetime), MAX(procedure_datetime)
FROM procedure_occurrence
WHERE provider_id IS NOT NULL AND procedure_concept_id = {extubationConceptId}
GROUP BY procedure_source_value
UNION ALL
SELECT 'self extubation in measurement', measurement_source_value, COUNT(*), COUNT(DISTINCT visit_occurrence_id), MIN(measurement_datetime), MAX(measurement_datetime)
FROM measurement
WHERE provider_id IS NOT NULL AND measurement_concept_id = 0 AND measurement_source_value = '{autoExtubationSource}'
GROUP BY measurement_source_value
UNION ALL
SELECT 'self extubation in observation', observation_source_value, COUNT(*), COUNT(DISTINCT visit_occurrence_id), MIN(observation_datetime), MAX(observation_datetime)
FROM observation
WHERE provider_id IS NOT NULL AND observation_concept_id = 0 AND observation_source_value = '{autoExtubationSource}'
GROUP BY observation_source_value
"""
itemOverview = client.query(overviewSql).to_dataframe()
print("\nOverview per item:")
print(itemOverview)


Overview per item:
                              item                       sourceValue  records  admissions                 firstDate                  lastDate
0                       gcs verbal             Beste verbale reactie   206008       14119 2005-12-31 01:15:00+00:00 2018-06-13 18:08:00+00:00
1                       gcs verbal                 V_EMV_NICE_Opname    67981        7239 2009-06-12 06:08:00+00:00 2018-06-13 20:23:00+00:00
2                       gcs verbal                  V_EMV_NICE_24uur    18790        7165 2009-06-12 19:26:00+00:00 2018-07-29 19:29:00+00:00
3                 extubation order                         Extuberen      510         499 2006-01-01 01:45:00+00:00 2013-02-03 01:19:00+00:00
4             gcs verbal intubated             Beste verbale reactie   101764        5959 2005-12-31 14:31:00+00:00 2017-05-06 07:58:00+00:00
5             gcs verbal intubated                 V_EMV_NICE_Opname     3794         554 2012-09-05 19:29:00+00:00 2016-10-15 0

#!!!!!!!!!!INVASIVE MODES!!!!!!!!!!!

In [ ]:
#distinct values ventilatie mode (set)
modeValuesSql = f"""SELECT value_source_value, value_as_concept_id, COUNT(*) AS records, COUNT(DISTINCT visit_occurrence_id) AS admissions
FROM measurement WHERE provider_id IS NOT NULL AND measurement_concept_id = {ventilationModeConceptId} AND measurement_source_value = 'Ventilatie Mode (Set)'
GROUP BY value_source_value, value_as_concept_id ORDER BY records DESC
"""
modeValues = client.query(modeValuesSql).to_dataframe()
print("\nValues of Ventilatie Mode (Set):")
print(modeValues)

#values of ventilatie mode (set) - hope this si correct but maybe ask prof? Or in PP
invasiveModes = ["PS/CPAP", "PC", "PC (No trig)", "PS/CPAP (trig)", "VC", "NAVA", "SIMV(PC)+PS", "SIMV(VC)+PS", "Bi Vente", "VC (No trig)", "VC (trig)", "PRVC (No trig)", "PRVC", "VS", "PRVC (trig)"]
nivModes = ["PS/CPAP in NIV", "PC in NIV"]


Values of Ventilatie Mode (Set):
   value_source_value  value_as_concept_id  records  admissions
0             PS/CPAP             45880365   770496       12112
1                 PC              36308445   274822       12540
2        PC (No trig)             36308445   118110        9298
3      PS/CPAP (trig)             45880365    63443        8118
4                 VC              36308445    20487        6018
5      PS/CPAP in NIV             45880365    16778        1325
6                NAVA             45880365    10672         122
7         SIMV(PC)+PS                 <NA>     5833         455
8         SIMV(VC)+PS             36310227     1132         103
9           PC in NIV             36308445      859         117
10           Bi Vente             45880365      628          51
11           Stand By             36310823      192         123
12       VC (No trig)             36308445       11          11
13          VC (trig)             36308445        8           3
14    

In [44]:
#how many admissions with an invasive mode
invasiveSql = f"""
SELECT COUNT(*) AS records, COUNT(DISTINCT visit_occurrence_id) AS admissions, COUNT(DISTINCT person_id) AS patients
FROM measurement
WHERE provider_id IS NOT NULL AND measurement_concept_id = {ventilationModeConceptId}  AND measurement_source_value = 'Ventilatie Mode (Set)' AND TRIM(value_source_value) IN {tuple(invasiveModes)}
"""
invasiveCount = client.query(invasiveSql).to_dataframe()
print("\nInvasive ventilation:")
print(invasiveCount)


Invasive ventilation:
   records  admissions  patients
0  1265656       15202     14100


!!!!!!!!TRACHEOSTOMY!!!!!!!!!!!

In [45]:
#distinct values of toedieningsweg
oxygenRouteValuesSql = """
SELECT value_source_value, COUNT(*) AS records, COUNT(DISTINCT visit_occurrence_id) AS admissions
FROM observation
WHERE provider_id IS NOT NULL AND observation_source_value = 'Toedieningsweg'
GROUP BY value_source_value ORDER BY records DESC
"""
oxygenRouteValues = client.query(oxygenRouteValuesSql).to_dataframe()
print("\nValues of Toedieningsweg:")
print(oxygenRouteValues)

#admissions with oxygen through a tracheostoma or a speaking cannula/valve <- maybe exclude as easier lets say in PP very clear so prof can see if correct
tracheostomySql = """
SELECT DISTINCT visit_occurrence_id FROM observation
WHERE provider_id IS NOT NULL AND observation_source_value = 'Toedieningsweg' AND TRIM(value_source_value) IN ('Trach.stoma', 'Spreekcanule', 'Spreekklepje')
"""
tracheostomyAdmissions = client.query(tracheostomySql).to_dataframe()
print(f"\nAdmissions with a tracheostomy: {len(tracheostomyAdmissions)}")

#same as invasiveSql without the tracheostomy admissions
noTracheostomySql = invasiveSql + f"""AND visit_occurrence_id NOT IN (SELECT visit_occurrence_id FROM ({tracheostomySql}) WHERE visit_occurrence_id IS NOT NULL)"""
noTracheostomyCount = client.query(noTracheostomySql).to_dataframe()
print("\nInvasive ventilation without tracheostomy:")
print(noTracheostomyCount)


Values of Toedieningsweg:
        value_source_value  records  admissions
0                  O2-bril   338421       18459
1                Kunstneus    99422        2967
2             Spreekklepje    56393         725
3                    Kapje    52649        4913
4   Non-Rebreathing masker    36844        3342
5             Spreekcanule    16066         521
6              Diep Nasaal    15729         933
7                  B.Lucht    12749        1883
8                 Kinnebak     9078         622
9                     CPAP     4480         396
10                  Nasaal     3560         572
11             Trach.stoma     1446          98
12               Nebulizer      261          41
13                  Guedel       83          30
14                Waterset       61          49
15                    Ambu       21          18
16                 DL-tube        1           1

Admissions with a tracheostomy: 951

Invasive ventilation without tracheostomy:
   records  admissions  pati

!!!!!!!!SURGERY!!!!!!!!!!!

In [46]:
#nice admission type per admission, planned surgery wins over emergency if both are there we dont want surgery namely
surgerySql = f"""
WITH admissionType AS (
  SELECT visit_occurrence_id,
    LOGICAL_OR(TRIM(value_source_value) = 'Geplande chirurgie') AS planned,
    LOGICAL_OR(TRIM(value_source_value) = 'Spoed chirurgie') AS emergency
  FROM observation
  WHERE provider_id IS NOT NULL AND observation_concept_id = 37154585 AND observation_source_value = 'NICE Opname type'
  GROUP BY visit_occurrence_id
),
ventilated AS (
  SELECT DISTINCT visit_occurrence_id, person_id
  FROM measurement
  WHERE provider_id IS NOT NULL AND measurement_concept_id = {ventilationModeConceptId} AND measurement_source_value = 'Ventilatie Mode (Set)' AND TRIM(value_source_value) IN {tuple(invasiveModes)} 
  AND visit_occurrence_id NOT IN (SELECT visit_occurrence_id FROM ({tracheostomySql}) WHERE visit_occurrence_id IS NOT NULL)
)
SELECT
  CASE
    WHEN admissionType.planned THEN 'planned surgery'
    WHEN admissionType.emergency THEN 'emergency surgery'
    WHEN admissionType.visit_occurrence_id IS NULL THEN 'no admission type'
    ELSE 'medical'
  END AS admissionGroup,
  COUNT(*) AS admissions,
  COUNT(DISTINCT ventilated.person_id) AS patients
FROM ventilated
LEFT JOIN admissionType ON admissionType.visit_occurrence_id = ventilated.visit_occurrence_id
GROUP BY admissionGroup ORDER BY admissions DESC
"""
surgeryCount = client.query(surgerySql).to_dataframe()
print("\ninvasive ventilation without tracheostomy per admission type:")
print(surgeryCount)


invasive ventilation without tracheostomy per admission type:
      admissionGroup  admissions  patients
0  no admission type        8801      8282
1    planned surgery        2855      2823
2            medical        1707      1643
3  emergency surgery        1001       979


In [48]:
#admissions per year and how many have no nice admission type because above we see that its a lot missing and if systematic we need other methods
niceYearSql = """
WITH niceAdmissions AS (
  SELECT DISTINCT visit_occurrence_id
  FROM observation
  WHERE provider_id IS NOT NULL AND observation_concept_id = 37154585 AND observation_source_value = 'NICE Opname type'
)
SELECT EXTRACT(YEAR FROM visit_occurrence.visit_start_datetime) AS year,
  COUNT(*) AS admissions,
  COUNTIF(niceAdmissions.visit_occurrence_id IS NULL) AS noAdmissionType
FROM visit_occurrence
LEFT JOIN niceAdmissions ON niceAdmissions.visit_occurrence_id = visit_occurrence.visit_occurrence_id
GROUP BY year ORDER BY year
"""
niceYear = client.query(niceYearSql).to_dataframe()
print(niceYear)

    year  admissions  noAdmissionType
0   2006        9436             9434
1   2007          69               69
2   2008          78               71
3   2009          66               48
4   2010          64               39
5   2011          39               26
6   2012          42               13
7   2013       13000             4338
8   2014         151               29
9   2015          92                6
10  2016          46                2
11  2017          20                0
12  2018           3                0


In [ ]:
#first rows visit_occurrence to see  admission info there is
visitSample = client.query("SELECT * FROM visit_occurrence LIMIT 5").to_dataframe()
print(visitSample)

   visit_occurrence_id  person_id  visit_concept_id visit_start_date      visit_start_datetime visit_end_date        visit_end_datetime  visit_type_concept_id  provider_id  care_site_id  \
0                16628      14352             32037       2007-09-19 2007-09-19 06:53:00+00:00     2007-11-07 2007-11-07 09:59:00+00:00                  32827            1             1   
1                 3822       3320             32037       2009-12-10 2009-12-10 17:04:00+00:00     2009-12-14 2009-12-14 23:10:00+00:00                  32827           23             2   
2                 6912       5965             32037       2013-06-16 2013-06-16 05:59:00+00:00     2013-06-16 2013-06-16 23:37:00+00:00                  32827           25             2   
3                10239       8828             32037       2013-06-14 2013-06-14 14:50:00+00:00     2013-06-16 2013-06-16 13:22:00+00:00                  32827           25             1   
4                 2528       2199             32037    

In [49]:
#all specialties in the provider table <- seems a good approach if it shows what is planned
providerSql = """
SELECT provider.provider_id, provider.specialty_source_value, concept.concept_name
FROM provider
LEFT JOIN concept ON concept.concept_id = provider.specialty_concept_id
ORDER BY provider.provider_id
"""
providers = client.query(providerSql).to_dataframe()
print(providers)

    provider_id          specialty_source_value                             concept_name
0             1                 Cardiochirurgie                          Cardiac Surgery
1             2                     Cardiologie                               Cardiology
2             3                    Gynaecologie                               Gynecology
3             4    Heelkunde Gastro-enterologie                          General Surgery
4             5      Heelkunde Longen/Oncologie                         Thoracic Surgery
5             6             Heelkunde Oncologie                        Surgical Oncology
6             7                     Hematologie                               Hematology
7             8      Intensive Care Volwassenen              Critical care (intensivist)
8             9                        Inwendig                        Internal Medicine
9            10            Keel, Neus & Oorarts                           Otolaryngology
10           11      

In [50]:
#ventilated admissions without tracheostomy per admitting specialty
specialtySql = f"""
WITH ventilated AS (
  SELECT DISTINCT visit_occurrence_id
  FROM measurement
  WHERE provider_id IS NOT NULL AND measurement_concept_id = {ventilationModeConceptId} AND measurement_source_value = 'Ventilatie Mode (Set)' AND TRIM(value_source_value) IN {tuple(invasiveModes)} AND visit_occurrence_id NOT IN (SELECT visit_occurrence_id FROM ({tracheostomySql}) WHERE visit_occurrence_id IS NOT NULL)
)
SELECT provider.provider_id, provider.specialty_source_value, COUNT(*) AS admissions
FROM ventilated
JOIN visit_occurrence ON visit_occurrence.visit_occurrence_id = ventilated.visit_occurrence_id
JOIN provider ON provider.provider_id = visit_occurrence.provider_id
GROUP BY provider.provider_id, provider.specialty_source_value ORDER BY admissions DESC
"""
specialties = client.query(specialtySql).to_dataframe()
print(specialties)

    provider_id        specialty_source_value  admissions
0             1               Cardiochirurgie        7158
1             2                   Cardiologie        1118
2            15                Neurochirurgie         841
3             8    Intensive Care Volwassenen         835
4            25                 Vaatchirurgie         666
5             4  Heelkunde Gastro-enterologie         657
6            23                 Traumatologie         550
7             9                      Inwendig         515
8            16                    Neurologie         405
9            10          Keel, Neus & Oorarts         224
10           11                    Longziekte         223
11            6           Heelkunde Oncologie         182
12            7                   Hematologie         166
13            5    Heelkunde Longen/Oncologie         149
14           20                    Orthopedie         101
15            3                  Gynaecologie          93
16           2

In [51]:
#current group invasive ventilation no tracheostomy then we decide the rest, this query can be reused as our current group...
ventilatedSql = f"""
SELECT DISTINCT visit_occurrence_id, person_id
FROM measurement
WHERE provider_id IS NOT NULL AND measurement_concept_id = {ventilationModeConceptId} AND measurement_source_value = 'Ventilatie Mode (Set)'
AND TRIM(value_source_value) IN {tuple(invasiveModes)} AND visit_occurrence_id NOT IN (SELECT visit_occurrence_id FROM ({tracheostomySql}) WHERE visit_occurrence_id IS NOT NULL)
"""
ventilated = client.query(ventilatedSql).to_dataframe()
print(f"admissions: {len(ventilated)}, patients: {ventilated['person_id'].nunique()}")

admissions: 14364, patients: 13548


In [54]:
#nice admission type per specialty, only admissions that have a nice type
specialtyNiceSql = f"""
WITH ventilated AS ({ventilatedSql}),
admissionType AS (
  SELECT visit_occurrence_id,
    LOGICAL_OR(TRIM(value_source_value) = 'Geplande chirurgie') AS planned,
    LOGICAL_OR(TRIM(value_source_value) = 'Spoed chirurgie') AS emergency
  FROM observation
  WHERE provider_id IS NOT NULL AND observation_concept_id = 37154585 AND observation_source_value = 'NICE Opname type'
  GROUP BY visit_occurrence_id
)
SELECT provider.specialty_source_value,
  COUNTIF(admissionType.planned) AS planned,
  COUNTIF(NOT admissionType.planned AND admissionType.emergency) AS emergency,
  COUNTIF(NOT admissionType.planned AND NOT admissionType.emergency) AS medical
FROM ventilated
JOIN admissionType ON admissionType.visit_occurrence_id = ventilated.visit_occurrence_id
JOIN visit_occurrence ON visit_occurrence.visit_occurrence_id = ventilated.visit_occurrence_id
JOIN provider ON provider.provider_id = visit_occurrence.provider_id
GROUP BY provider.specialty_source_value ORDER BY planned + emergency + medical DESC
"""
specialtyNice = client.query(specialtyNiceSql).to_dataframe()
print(specialtyNice)

          specialty_source_value  planned  emergency  medical
0                Cardiochirurgie     2480        185       53
1                    Cardiologie       29         10      515
2                 Neurochirurgie       37        251       43
3                  Traumatologie        9        153      135
4   Heelkunde Gastro-enterologie       82        121       64
5                       Inwendig        1         11      229
6                     Neurologie        1         21      182
7                  Vaatchirurgie       71        102       24
8                     Longziekte        7          8      129
9           Keel, Neus & Oorarts       11         42       47
10                   Hematologie        0          2       78
11    Intensive Care Volwassenen       16          8       47
12      Maag-,Darm-,Leverziekten        2          3       50
13           Heelkunde Oncologie       21         14       14
14                    Nefrologie        6          7       29
15      

In [ ]:
#NEW current group invasive ventilation no tracheostomy no cardiac surgery no planned surgery 
# (see above table cardiac mostly planned others have few planned so a bit of a bummer but cant delete those)
ventilatedSql = f"""
SELECT DISTINCT visit_occurrence_id, person_id
FROM measurement
WHERE provider_id IS NOT NULL AND measurement_concept_id = {ventilationModeConceptId} AND measurement_source_value = 'Ventilatie Mode (Set)'
AND TRIM(value_source_value) IN {tuple(invasiveModes)} AND visit_occurrence_id NOT IN (SELECT visit_occurrence_id FROM ({tracheostomySql}) WHERE visit_occurrence_id IS NOT NULL)
AND visit_occurrence_id NOT IN (
  SELECT visit_occurrence_id FROM visit_occurrence
  WHERE provider_id = 1 -- cardiochirurgie
  )
AND visit_occurrence_id NOT IN (
  SELECT visit_occurrence_id FROM observation
  WHERE provider_id IS NOT NULL
  AND observation_concept_id = 37154585
  AND observation_source_value = 'NICE Opname type'
  AND TRIM(value_source_value) = 'Geplande chirurgie'
  AND visit_occurrence_id IS NOT NULL
)
"""
ventilated = client.query(ventilatedSql).to_dataframe()
print(f"admissions: {len(ventilated)}, patients: {ventilated['person_id'].nunique()}")

admissions: 6831, patients: 6373


OUTCOMES (so above is group selection mostly below is the outcome variable selection/computation)

In [57]:
#death per admission in our group whether was during or after admission
deathSql = f"""
WITH ventilated AS ({ventilatedSql})
SELECT COUNT(*) AS admissions,
  COUNTIF(death.person_id IS NOT NULL) AS withDeath,
  COUNTIF(death.death_date <= visit_occurrence.visit_end_date) AS diedDuringAdmission,
  COUNTIF(death.death_date > visit_occurrence.visit_end_date) AS diedAfterAdmission,
  COUNTIF(death.death_date > visit_occurrence.visit_end_date AND death.death_date <= DATE_ADD(visit_occurrence.visit_end_date, INTERVAL 7 DAY)) AS diedWithin7DaysAfter
FROM ventilated
JOIN visit_occurrence ON visit_occurrence.visit_occurrence_id = ventilated.visit_occurrence_id
LEFT JOIN death ON death.person_id = ventilated.person_id
"""
deaths = client.query(deathSql).to_dataframe()
print(deaths)

   admissions  withDeath  diedDuringAdmission  diedAfterAdmission  diedWithin7DaysAfter
0        6831       3263                 1793                1470                   168


In [58]:
#next admissio same patient within 7 days after admission ended (ventilated)
readmissionSql = f"""
WITH ventilated AS ({ventilatedSql})
SELECT COUNT(DISTINCT ventilated.visit_occurrence_id) AS admissions,
  COUNT(DISTINCT IF(nextVisit.visit_occurrence_id IS NOT NULL, ventilated.visit_occurrence_id, NULL)) AS readmittedWithin7Days,
  COUNT(DISTINCT IF(nextVisit.visit_start_datetime <= TIMESTAMP_ADD(currentVisit.visit_end_datetime, INTERVAL 1 HOUR), ventilated.visit_occurrence_id, NULL)) AS nextWithin1Hour
FROM ventilated
JOIN visit_occurrence currentVisit ON currentVisit.visit_occurrence_id = ventilated.visit_occurrence_id
LEFT JOIN visit_occurrence nextVisit
  ON nextVisit.person_id = currentVisit.person_id
  AND nextVisit.visit_start_datetime >= currentVisit.visit_end_datetime
  AND nextVisit.visit_start_datetime <= TIMESTAMP_ADD(currentVisit.visit_end_datetime, INTERVAL 7 DAY)
"""
readmissions = client.query(readmissionSql).to_dataframe()
print(readmissions)

   admissions  readmittedWithin7Days  nextWithin1Hour
0        6831                    422               58


EXTUBATIONS

In [ ]:
#all invasive ventilation records of our group, sorted per patient in time mostly used later maybe pp
invasiveRecordsSql = f"""
WITH ventilated AS ({ventilatedSql})
SELECT measurement.person_id, measurement.visit_occurrence_id, measurement.measurement_datetime,
  TRIM(measurement.value_source_value) AS mode
FROM measurement
JOIN ventilated ON ventilated.visit_occurrence_id = measurement.visit_occurrence_id
WHERE measurement.provider_id IS NOT NULL AND measurement.measurement_concept_id = {ventilationModeConceptId} AND measurement.measurement_source_value = 'Ventilatie Mode (Set)'
AND TRIM(measurement.value_source_value) IN {tuple(invasiveModes)}
ORDER BY measurement.person_id, measurement.measurement_datetime
"""
invasiveRecords = client.query(invasiveRecordsSql).to_dataframe()
print(f"records: {len(invasiveRecords)}")

records: 706712


In [ ]:
#oxygen route records of our group mostly used later and more often maybe pp
oxygenRecordsSql = f"""
WITH ventilated AS ({ventilatedSql})
SELECT observation.person_id, observation.observation_datetime, TRIM(observation.value_source_value) AS route
FROM observation
JOIN ventilated ON ventilated.visit_occurrence_id = observation.visit_occurrence_id
WHERE observation.provider_id IS NOT NULL
AND observation.observation_source_value = 'Toedieningsweg'
"""
oxygenRecords = client.query(oxygenRecordsSql).to_dataframe()
print(f"oxygen records: {len(oxygenRecords)}")

oxygen records: 210543


In [61]:
#ventilator and oxygen records together in one timeline per patient
ventilatorRows = invasiveRecords[["person_id", "measurement_datetime", "mode"]]
ventilatorRows.columns = ["person_id", "time", "value"]
ventilatorRows = ventilatorRows.assign(source="ventilator")
oxygenRows = oxygenRecords[["person_id", "observation_datetime", "route"]]
oxygenRows.columns = ["person_id", "time", "value"]
oxygenRows = oxygenRows.assign(source="oxygen")
allTimelines = pd.concat([ventilatorRows, oxygenRows]).sort_values(["person_id", "time"])
print(f"timeline rows: {len(allTimelines)}")

timeline rows: 917255


In [62]:
#tube ventilator or oxygen through the tube (kunstneus/ waterset/ dl-tube) and noTube = oxygen without a tube
allTimelines["state"] = "noTube"
allTimelines.loc[allTimelines["source"] == "ventilator", "state"] = "tube"
allTimelines.loc[allTimelines["value"].isin(["Kunstneus", "Waterset", "DL-tube"]), "state"] = "tube"

stateSwitches = []
previousRow = None
for row in allTimelines.itertuples():
    if previousRow is not None and row.person_id == previousRow.person_id and row.state != previousRow.state:
        stateSwitches.append({
            "person_id": row.person_id,
            "direction": previousRow.state + " -> " + row.state,
            "lastBefore": previousRow.time,
            "firstAfter": row.time,
            "valueBefore": previousRow.value,
            "valueAfter": row.value,
        })
    previousRow = row
stateSwitches = pd.DataFrame(stateSwitches)
stateSwitches["hoursBetween"] = (stateSwitches["firstAfter"] - stateSwitches["lastBefore"]).dt.total_seconds() / 3600

print(stateSwitches["direction"].value_counts())
print(stateSwitches.groupby("direction")["hoursBetween"].describe())
print(stateSwitches[stateSwitches["direction"] == "tube -> noTube"]["valueBefore"].value_counts().head(10))

direction
tube -> noTube    5666
noTube -> tube    2044
Name: count, dtype: int64
                 count        mean          std  min  25%  50%  75%            max
direction                                                                         
noTube -> tube  2044.0  718.840623  4866.072592  0.0  1.0  2.0  6.0  100491.666667
tube -> noTube  5666.0    7.727627   339.200183  0.0  0.5  1.0  1.0   22081.000000
valueBefore
PS/CPAP           3808
Kunstneus         1068
PC (No trig)       272
PS/CPAP (trig)     241
PC                 200
VC                  29
SIMV(PC)+PS         15
Bi Vente            12
Waterset             9
SIMV(VC)+PS          5
Name: count, dtype: int64


In [63]:
#per extubation the time until the patient is back on a tube (reintubation), if ever
#!!!!Within 1 hour reintubation real? maybe ask?
extubationList = []
for row in stateSwitches.itertuples():
    if row.direction == "tube -> noTube":
        extubationList.append({
            "person_id": row.person_id,
            "extubationTime": row.lastBefore,
            "valueBefore": row.valueBefore,
            "reintubationTime": None,
        })
    #a switch back to tube belongs to the last extubation of the same patient
    elif row.direction == "noTube -> tube" and len(extubationList) > 0 and extubationList[-1]["person_id"] == row.person_id:
        extubationList[-1]["reintubationTime"] = row.firstAfter
extubations = pd.DataFrame(extubationList)
extubations["reintubationTime"] = pd.to_datetime(extubations["reintubationTime"], utc=True)
extubations["hoursToReintubation"] = (extubations["reintubationTime"] - extubations["extubationTime"]).dt.total_seconds() / 3600

print(f"extubations: {len(extubations)}, never back on a tube: {extubations['hoursToReintubation'].isna().sum()}")
print(pd.cut(extubations["hoursToReintubation"], bins=[0, 1, 2, 6, 24, 48, 168, 1000000], include_lowest=True).value_counts().sort_index())

extubations: 5666, never back on a tube: 4280
hoursToReintubation
(-0.001, 1.0]         105
(1.0, 2.0]             99
(2.0, 6.0]            163
(6.0, 24.0]           272
(24.0, 48.0]          166
(48.0, 168.0]         282
(168.0, 1000000.0]    299
Name: count, dtype: int64


LOOK AT IF EXTUBATION DUE TO PALIATIVE CARE

In [64]:
terminalPattern = "beperking|palliat|comfort|reanim|terminal|staken|withdraw|limitation"
inDescription = dictionary["source_code_description"].str.contains(terminalPattern, case=False, na=False)
inConceptName = dictionary["concept_name"].str.contains(terminalPattern, case=False, na=False)
print(dictionary[inDescription | inConceptName][["source_vocabulary_id", "source_code", "source_code_description", "concept_name", "source_frequency"]])

       source_vocabulary_id                                        source_code                            source_code_description                                       concept_name  source_frequency
2679        AUMC Text Value  leucocytose 43,9 onbegrepen pulm zwak waarbij ...  leucocytose 43,9 onbegrepen pulm zwak waarbij ...                                                NaN                 1
3580              AUMC List                                               7969                                    RedenStakenCVVH                                No matching concept              1872
3599              AUMC List                                              16101                       Reanimatie Ondernomen acties                                No matching concept               300
3601              AUMC List                                              16102                                 Reanimatie Oorzaak                                     Cardiac arrest               185
3605 

In [65]:
#treatment policy per admission #!!Check if DNR also counts as do not reintubate with prof
policySql = f"""
WITH ventilated AS ({ventilatedSql})
SELECT TRIM(observation.value_source_value) AS policy, COUNT(DISTINCT observation.visit_occurrence_id) AS admissions
FROM observation
JOIN ventilated ON ventilated.visit_occurrence_id = observation.visit_occurrence_id
WHERE observation.provider_id IS NOT NULL AND observation.observation_concept_id = 1761496 AND observation.observation_source_value = 'Beleid'
GROUP BY policy ORDER BY policy
"""
policies = client.query(policySql).to_dataframe()
print(policies)

  policy  admissions
0      I        1755
1     II        1710
2    III         467


In [66]:
#death per patient in our group
deathsSql = f"""
WITH ventilated AS ({ventilatedSql})
SELECT DISTINCT death.person_id, death.death_datetime
FROM death
JOIN ventilated ON ventilated.person_id = death.person_id
"""
patientDeaths = client.query(deathsSql).to_dataframe()
print(f"patients with death: {len(patientDeaths)}, without a time: {patientDeaths['death_datetime'].isna().sum()}")


patients with death: 3021, without a time: 0


In [67]:
#beleid records with time, to know the policy at each extubation
policyRecordsSql = f"""
WITH ventilated AS ({ventilatedSql})
SELECT observation.person_id, observation.observation_datetime AS policyTime, TRIM(observation.value_source_value) AS policy
FROM observation
JOIN ventilated ON ventilated.visit_occurrence_id = observation.visit_occurrence_id
WHERE observation.provider_id IS NOT NULL AND observation.observation_concept_id = 1761496 AND observation.observation_source_value = 'Beleid'
"""
policyRecords = client.query(policyRecordsSql).to_dataframe()
print(f"beleid records: {len(policyRecords)}")

beleid records: 8396


In [ ]:
#time from extubation to death
extubationOutcomes = extubations.merge(patientDeaths, on="person_id", how="left")
extubationOutcomes["hoursToDeath"] = (extubationOutcomes["death_datetime"] - extubationOutcomes["extubationTime"]).dt.total_seconds() / 3600

#policy at moment of extubation is last beleid record before
policyRecords["person_id"] = policyRecords["person_id"].astype("int64")
extubationOutcomes = extubationOutcomes.sort_values("extubationTime")
policyRecords = policyRecords.sort_values("policyTime")
extubationOutcomes = pd.merge_asof(extubationOutcomes, policyRecords, left_on="extubationTime", right_on="policyTime", by="person_id", direction="backward")
extubationOutcomes["policy"] = extubationOutcomes["policy"].fillna("none")

#died within 7 days and not reintubated before death
diedWithoutReintubation = (extubationOutcomes["hoursToDeath"] <= 168) & (extubationOutcomes["reintubationTime"].isna() | (extubationOutcomes["reintubationTime"] > extubationOutcomes["death_datetime"]))
diedExtubations = extubationOutcomes[diedWithoutReintubation]
print(f"died within 7 days without reintubation: {len(diedExtubations)}")
print(pd.crosstab(pd.cut(diedExtubations["hoursToDeath"], bins=[0, 6, 24, 72, 168], include_lowest=True), diedExtubations["policy"]))
#Below table shows sadly we cant say if died without reintubation that it was paliative as group 3 
# does not have biggest share. (Also small groups as a lot has been fixed already due to above criteria)

died within 7 days without reintubation: 230
policy          I  II  III  none
hoursToDeath                    
(-0.001, 6.0]   1   6    5     2
(6.0, 24.0]     1  19    4     4
(24.0, 72.0]    4  39    8     5
(72.0, 168.0]  17  44   11    17


In [ ]:
#comfort care at extubation leave out
goldStandard = extubationOutcomes[extubationOutcomes["policy"] != "III"].copy()

#failure defined as back on a tube or death within 7 days. 48h version as one paper does that maybe to look at sensitive cases more?
goldStandard["failure7Days"] = (goldStandard["hoursToReintubation"] <= 168) | (goldStandard["hoursToDeath"] <= 168)
goldStandard["failure48Hours"] = (goldStandard["hoursToReintubation"] <= 48) | (goldStandard["hoursToDeath"] <= 48)

print(f"extubations: {len(goldStandard)}, patients: {goldStandard['person_id'].nunique()}")
print(goldStandard["failure7Days"].value_counts())
print(goldStandard["failure48Hours"].value_counts())

#admission of each extubation is admission of the last invasive record before it
invasiveRecords["person_id"] = invasiveRecords["person_id"].astype("int64")
recordAdmissions = invasiveRecords[["person_id", "measurement_datetime", "visit_occurrence_id"]].sort_values("measurement_datetime")
goldStandard = goldStandard.sort_values("extubationTime")
goldStandard = pd.merge_asof(goldStandard, recordAdmissions, left_on="extubationTime", right_on="measurement_datetime", by="person_id", direction="backward")
goldStandard = goldStandard.drop(columns="measurement_datetime")

#attempt number within the admission 1 = first extubation etc I removed ones above 5 lets name it in PP were few records and seems unrealistic (or pat with tracheactomy as is easy to remove)
goldStandard = goldStandard.sort_values(["visit_occurrence_id", "extubationTime"])
goldStandard["attemptNumber"] = goldStandard.groupby("visit_occurrence_id").cumcount() + 1

#patients with 5 or more extubations one admission left out notd realistic weaning course I think PP
goldStandard = goldStandard[~goldStandard["person_id"].isin(
    goldStandard[goldStandard["attemptNumber"] >= 5]["person_id"])]

print(f"extubations: {len(goldStandard)}, patients: {goldStandard['person_id'].nunique()}")

print(goldStandard["attemptNumber"].value_counts().sort_index())
print(goldStandard.groupby("attemptNumber")["failure7Days"].mean())

extubations: 5555, patients: 4466
failure7Days
False    4346
True     1209
Name: count, dtype: int64
failure48Hours
False    4707
True      848
Name: count, dtype: int64
extubations: 5369, patients: 4443
attemptNumber
1.0    4760
2.0     508
3.0      84
4.0      12
Name: count, dtype: int64
attemptNumber
1.0    0.186345
2.0    0.244094
3.0    0.273810
4.0    0.250000
Name: failure7Days, dtype: float64
